In [72]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mikeytracegod_lung_cancer_risk_dataset_path = kagglehub.dataset_download('mikeytracegod/lung-cancer-risk-dataset')

print('Data source import complete.', mikeytracegod_lung_cancer_risk_dataset_path)


Data source import complete. C:\Users\dsapu\.cache\kagglehub\datasets\mikeytracegod\lung-cancer-risk-dataset\versions\1


In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import optuna
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, FunctionTransformer, PolynomialFeatures, QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression

from torch import nn
import torch
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.optim as optim
# !pip install -q pytorch-optimizer
import pytorch_optimizer as optim1


from torchmetrics.functional import mean_squared_error, r2_score
from pytorch_lightning import LightningModule
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.loggers import CSVLogger
import pytorch_lightning as pl
from torchmetrics.regression import MeanAbsoluteError, MeanSquaredError, R2Score
from torchmetrics.classification import F1Score, confusion_matrix

In [74]:
class MyDataset(Dataset):
    # 1. UBAH DI SINI: Tambahkan parameter untuk menerima data
    def __init__(self, features, labels):
        # 2. Simpan data yang diterima sebagai atribut class
        self.features = features
        self.labels = labels

    # 3. Metode __len__ harus mengembalikan jumlah total sampel
    def __len__(self):
        return len(self.features)

    # 4. Metode __getitem__ harus mengambil satu sampel berdasarkan indeks
    def __getitem__(self, idx):
        # Ambil fitur dan label pada indeks 'idx'
        x = self.features[idx]
        y = self.labels[idx]
        
        # Pastikan outputnya sudah dalam bentuk tensor (jika belum)
        # Jika X_train dan y_train sudah tensor, baris di bawah tidak wajib
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        if not isinstance(y, torch.Tensor):
            y = torch.tensor(y, dtype=torch.float32)
            
        return x, y

In [ ]:
class NNModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(NNModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Mish(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 128),
            nn.Mish(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 128),
            nn.Mish(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),

            nn.Linear(128, 32),
            nn.Mish(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.1),

            nn.Linear(32, 16),
            nn.Mish(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.1),

            nn.Linear(16, output_dim)
        )
    def forward(self, x):
        return self.model(x)
    
class PLNN(LightningModule):
    def __init__(self, input_size, class_weight1, num_classes, learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters() 
        self.model = NNModel(input_dim=input_size, output_dim=num_classes)
        self.criterion = nn.CrossEntropyLoss(weight=class_weight1)
        self.mse = MeanSquaredError(squared=False, num_outputs=num_classes)
        self.f1 = F1Score('binary', average='weighted')
    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs) 
        loss = self.criterion(outputs, labels)
        mse = self.mse(outputs, labels)
        f1 = self.f1(outputs, labels)

        self.log('train_loss', loss, on_epoch=True, prog_bar=True)
        self.log('train_f1', f1.mean(), on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        mse = self.mse(outputs, labels)
        f1 = self.f1(outputs, labels)

        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_f1', f1.mean(), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = optim1.AdamW(self.parameters(), lr=self.hparams.learning_rate)
        return optimizer

    def compute_metrics(self, outputs, labels, prefix="val", num_classes=2):
        """
        Hitung metric per loader dengan torchmetrics.functional.
        """
        # Pastikan tensor sudah di-device yang sama
        loss = self.criterion(outputs, labels)

        # r2_score & mse dari torchmetrics.functional
        r2_per_label = r2_score(outputs, labels, multioutput="raw_values")
        mse_per_label = mean_squared_error(outputs, labels, num_outputs=num_classes)
        rmse_per_label = torch.sqrt(mse_per_label)

        return {
            f"{prefix}_loss": loss.item(),
            f"{prefix}_rmse_label_1": rmse_per_label[0].item(),
            f"{prefix}_rmse_label_2": rmse_per_label[1].item(),
            f"{prefix}_rmse_avg": rmse_per_label.mean().item(),
            f"{prefix}_r2_label_1": r2_per_label[0].item(),
            f"{prefix}_r2_label_2": r2_per_label[1].item(),
            f"{prefix}_r2_avg": r2_per_label.mean().item()
        }

    def evaluate_loader(self, loader, prefix="val"):
        """
        Evaluasi seluruh data dalam satu DataLoader.
        """
        self.eval()
        all_outputs, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                inputs, labels, *_ = batch
                outputs = self(inputs)
                all_outputs.append(outputs)
                all_labels.append(labels)

        all_outputs = torch.cat(all_outputs, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        return self.compute_metrics(all_outputs, all_labels, prefix)

In [76]:
df = pd.read_csv("lung_cancer_dataset.csv", index_col='patient_id')
df

,age,gender,pack_years,radon_exposure,asbestos_exposure,secondhand_smoke_exposure,copd_diagnosis,alcohol_consumption,family_history,lung_cancer
patient_id,,,,,,,,,,
100000,69,Male,66.025244,High,No,No,Yes,Moderate,No,No
100001,32,Female,12.780800,High,No,Yes,Yes,Moderate,Yes,Yes
100002,89,Female,0.408278,Medium,Yes,Yes,Yes,NaN,No,Yes
100003,78,Female,44.065232,Low,No,Yes,No,Moderate,No,Yes
100004,38,Female,44.432440,Medium,Yes,No,Yes,NaN,Yes,Yes
...,...,...,...,...,...,...,...,...,...,...
149995,81,Female,9.386431,Medium,No,Yes,No,Moderate,No,Yes
149996,28,Male,99.471718,Medium,No,Yes,No,Moderate,No,Yes
149997,90,Male,14.349722,Low,Yes,Yes,No,Heavy,Yes,Yes


In [77]:
X = df.drop(columns='lung_cancer', axis=1)
y = df.lung_cancer

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [78]:
num = [i for i in X_train.columns if X_train[i].dtype != 'object']
cat = [i for i in X_train.columns if i not in num]
num, cat

(['age', 'pack_years'],
 ['gender',
  'radon_exposure',
  'asbestos_exposure',
  'secondhand_smoke_exposure',
  'copd_diagnosis',
  'alcohol_consumption',
  'family_history'])

In [79]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer()),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2))
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num),
    ('cat', cat_pipe, cat)
])

In [80]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [81]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.map({'Yes': 1, 'No': 0}).values, dtype=torch.float32)
y_test = torch.tensor(y_test.map({'Yes': 1, 'No': 0}).values, dtype=torch.float32)

train_set = MyDataset(X_train, y_train)
test_set = MyDataset(X_test, y_test)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

In [82]:
feature, target = next(iter(train_loader))
feature.shape, target.shape

(torch.Size([128, 21]), torch.Size([128]))

In [86]:
ylist = y_train.tolist()
y0 = [i for i in ylist if i < 1]
y1 = [i for i in ylist if i not in y0]
weight0 = (len(y0)+len(y1))/len(y0)
weight1 = (len(y0)+len(y1))/len(y1)
class_weight = torch.tensor([weight0, weight1])
print(class_weight)

tensor([3.1977, 1.4550])


SyntaxError: invalid syntax (3361673808.py, line 1)